# ASL Hand Sign (Sign Language MNIST) — Train + Webcam (MediaPipe ROI)

โน้ตบุ๊กนี้ทำตั้งแต่ต้น:
1) โหลดชุดข้อมูล Sign Language MNIST (CSV)
2) เทรน CNN
3) เซฟโมเดล
4) รันกล้องจริงแบบ real-time โดยใช้ MediaPipe ครอปมือ (ROI) แล้วค่อยทำนาย

> **สำคัญ:** เซลล์เว็บแคมต้องรันบนเครื่องคุณ (Jupyter Local) เพราะต้องเข้าถึงกล้อง


## 0) สร้าง venv (ทำใน Terminal ก่อนเปิด Jupyter)

### Windows (PowerShell)
```powershell
py -m venv .venv
.\.venv\Scripts\Activate.ps1
python -m pip install --upgrade pip
pip install numpy pandas scikit-learn tensorflow opencv-python mediapipe matplotlib
pip install jupyter
jupyter notebook
```

### macOS / Linux
```bash
python3 -m venv .venv
source .venv/bin/activate
python -m pip install --upgrade pip
pip install numpy pandas scikit-learn tensorflow opencv-python mediapipe matplotlib
pip install jupyter
jupyter notebook
```

โครงสร้างไฟล์:
```
asl_hand_sign/
  data/
    sign_mnist_train.csv
    sign_mnist_test.csv
  artifacts/
  ASL_Sign_MNIST_Train_and_Webcam.ipynb
```


## 1) Import + Config


In [2]:
import os
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

DATA_DIR = "data"
TRAIN_CSV = os.path.join(DATA_DIR, "sign_mnist_train.csv")
TEST_CSV  = os.path.join(DATA_DIR, "sign_mnist_test.csv")

OUT_DIR = "artifacts"
os.makedirs(OUT_DIR, exist_ok=True)

MODEL_PATH = os.path.join(OUT_DIR, "asl_cnn.keras")

print("TensorFlow:", tf.__version__)
print("Train exists?", os.path.exists(TRAIN_CSV))
print("Test exists?", os.path.exists(TEST_CSV))


ModuleNotFoundError: No module named 'pandas'

## 2) โหลดข้อมูลจาก CSV


In [ ]:
def load_csv(path: str):
    df = pd.read_csv(path)
    y = df["label"].values.astype(np.int64)
    X = df.drop(columns=["label"]).values.astype(np.float32)
    X = X.reshape(-1, 28, 28, 1)
    X /= 255.0
    return X, y

X_train, y_train = load_csv(TRAIN_CSV)
X_test, y_test = load_csv(TEST_CSV)

num_classes = int(max(y_train.max(), y_test.max()) + 1)
print("X_train:", X_train.shape, "y_train:", y_train.shape)
print("X_test:", X_test.shape, "y_test:", y_test.shape)
print("num_classes:", num_classes)
print("labels in train:", sorted(set(y_train))[:10], "...", sorted(set(y_train))[-10:])


X_train: (27455, 28, 28, 1) y_train: (27455,)
X_test: (7172, 28, 28, 1) y_test: (7172,)
num_classes: 25
labels in train: [np.int64(0), np.int64(1), np.int64(2), np.int64(3), np.int64(4), np.int64(5), np.int64(6), np.int64(7), np.int64(8), np.int64(10)] ... [np.int64(15), np.int64(16), np.int64(17), np.int64(18), np.int64(19), np.int64(20), np.int64(21), np.int64(22), np.int64(23), np.int64(24)]


## 3) สร้างโมเดล CNN


In [ ]:
def build_model(num_classes: int):
    model = keras.Sequential([
        layers.Input(shape=(28, 28, 1)),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.Conv2D(32, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.Conv2D(64, 3, padding="same", activation="relu"),
        layers.MaxPooling2D(),
        layers.Dropout(0.25),

        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(num_classes, activation="softmax"),
    ])
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model

model = build_model(num_classes)
model.summary()


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 28, 28, 32)     │           320 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 28, 28, 32)     │         9,248 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ (None, 14, 14, 32)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 14, 14, 64)     │        18,496 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 14, 14, 64)     │        36,928 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 7, 7, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 3136)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 128)            │       401,536 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 25)             │         3,225 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 469,753 (1.79 MB)

 Trainable params: 469,753 (1.79 MB)

 Non-trainable params: 0 (0.00 B)

## 4) เทรน + เซฟโมเดล


In [ ]:
callbacks = [
    keras.callbacks.ModelCheckpoint(MODEL_PATH, save_best_only=True, monitor="val_accuracy", mode="max"),
    keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True, monitor="val_accuracy", mode="max"),
]

history = model.fit(
    X_train, y_train,
    validation_split=0.1,
    epochs=30,
    batch_size=128,
    callbacks=callbacks,
    verbose=1,
)

print("Saved best model to:", MODEL_PATH)


Epoch 1/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 8s 37ms/step - accuracy: 0.4249 - loss: 1.8509 - val_accuracy: 0.9370 - val_loss: 0.2591
Epoch 2/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 7s 33ms/step - accuracy: 0.8754 - loss: 0.3658 - val_accuracy: 0.9949 - val_loss: 0.0299
Epoch 3/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.9457 - loss: 0.1556 - val_accuracy: 0.9993 - val_loss: 0.0063
Epoch 4/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 7s 34ms/step - accuracy: 0.9691 - loss: 0.0916 - val_accuracy: 1.0000 - val_loss: 8.1307e-04
Epoch 5/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 7s 34ms/step - accuracy: 0.9769 - loss: 0.0682 - val_accuracy: 1.0000 - val_loss: 9.7131e-04
Epoch 6/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 7s 34ms/step - accuracy: 0.9834 - loss: 0.0493 - val_accuracy: 1.0000 - val_loss: 4.9862e-04
Epoch 7/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 6s 33ms/step - accuracy: 0.9854 - loss: 0.0452 - val_accuracy: 1.0000 - val_loss: 1.7847e-04
Epoch 8/30
194/194 ━━━━━━━━━━━━━━━━━━━━ 6s 32ms/step - accuracy: 0.9866 - loss: 0

## 5) ประเมินผล (Report + Confusion Matrix)


In [ ]:
best_model = tf.keras.models.load_model(MODEL_PATH)
test_loss, test_acc = best_model.evaluate(X_test, y_test, verbose=0)
print(f"Test accuracy: {test_acc:.4f} | Test loss: {test_loss:.4f}")

y_pred = np.argmax(best_model.predict(X_test, verbose=0), axis=1)
print("\nClassification report:")
print(classification_report(y_test, y_pred, digits=4))
print("Confusion matrix:")
print(confusion_matrix(y_test, y_pred))


Test accuracy: 0.9399 | Test loss: 0.1914

Classification report:
              precision    recall  f1-score   support

           0     0.9735    1.0000    0.9866       331
           1     1.0000    0.9907    0.9953       432
           2     1.0000    1.0000    1.0000       310
           3     1.0000    1.0000    1.0000       245
           4     0.9978    0.9157    0.9550       498
           5     1.0000    1.0000    1.0000       247
           6     0.9169    0.8879    0.9022       348
           7     1.0000    0.9472    0.9729       436
           8     0.9248    0.8542    0.8881       288
          10     0.8378    0.9366    0.8845       331
          11     0.9952    1.0000    0.9976       209
          12     0.9221    0.9010    0.9114       394
          13     0.9416    0.8316    0.8832       291
          14     1.0000    0.9797    0.9897       246
          15     0.9455    1.0000    0.9720       347
          16     1.0000    1.0000    1.0000       164
          17   

## 6) Label → ตัวอักษร (Mapping)

โดยทั่วไป Sign Language MNIST มักมี 24 คลาส (ไม่รวม J, Z เพราะต้องเคลื่อนไหว)
mapping ที่พบบ่อยคือ: 0=A,1=B,...,8=I,9=K,...,23=Y

> ถ้าคุณอยากเช็กว่าคลาสมีอะไรจริง ๆ ให้ดู `sorted(set(y_train))` และเทียบจำนวนคลาส


In [ ]:
IDX_TO_CHAR = {
    0: "A", 1: "B", 2: "C", 3: "D", 4: "E",
    5: "F", 6: "G", 7: "H", 8: "I",
    9: "K", 10: "L", 11: "M", 12: "N", 13: "O",
    14: "P", 15: "Q", 16: "R", 17: "S", 18: "T",
    19: "U", 20: "V", 21: "W", 22: "X", 23: "Y",
}

def label_to_char(label: int) -> str:
    return IDX_TO_CHAR.get(int(label), f"#{label}")

print([label_to_char(i) for i in range(min(num_classes, 24))])


['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H', 'I', 'K', 'L', 'M', 'N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y']


## 7) Real-time Webcam (MediaPipe ครอปมือ → 28×28 → Predict)

### ทำให้ “มือถือเป็นเว็บแคม” (สำหรับเทส)
- วิธีง่ายสุด: **DroidCam** หรือ **Iriun Webcam** (ลงในมือถือ + ลงใน Windows)
- เชื่อม Wi‑Fi เดียวกันหรือ USB
- จากนั้น OpenCV จะเห็นเป็นกล้องใหม่ → เปลี่ยน `CAMERA_INDEX`

### หมายเหตุ
- ต้องรันเซลล์นี้บนเครื่องคุณที่มีกล้อง
- ถ้ากล้องไม่ขึ้น ให้ลอง `CAMERA_INDEX = 1,2,3`


In [ ]:
# import cv2
# import mediapipe as mp
# from collections import deque

# CAMERA_INDEX = 0   # ลอง 0,1,2,3 ถ้าไม่เจอ
# SMOOTH_N = 7

# def preprocess_roi(roi_bgr: np.ndarray) -> np.ndarray:
#     gray = cv2.cvtColor(roi_bgr, cv2.COLOR_BGR2GRAY)
#     gray = cv2.equalizeHist(gray)
#     img = cv2.resize(gray, (28, 28), interpolation=cv2.INTER_AREA)
#     x = img.astype(np.float32) / 255.0
#     return x.reshape(1, 28, 28, 1)

# def majority_vote(labels):
#     if not labels:
#         return None
#     vals, counts = np.unique(np.array(labels), return_counts=True)
#     return int(vals[np.argmax(counts)])

# mp_hands = mp.solutions.hands
# hands = mp_hands.Hands(
#     static_image_mode=False,
#     max_num_hands=1,
#     model_complexity=1,
#     min_detection_confidence=0.6,
#     min_tracking_confidence=0.6,
# )

# cap = cv2.VideoCapture(CAMERA_INDEX)
# if not cap.isOpened():
#     raise RuntimeError("เปิดกล้องไม่สำเร็จ: ลองเปลี่ยน CAMERA_INDEX เป็น 1/2/3")

# pred_queue = deque(maxlen=SMOOTH_N)

# print("Press Q to quit")
# while True:
#     ret, frame = cap.read()
#     if not ret:
#         break

#     frame = cv2.flip(frame, 1)
#     h, w, _ = frame.shape

#     rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
#     res = hands.process(rgb)

#     label_text = "No hand"
#     conf_text = ""

#     if res.multi_hand_landmarks:
#         lm = res.multi_hand_landmarks[0].landmark
#         xs = [p.x for p in lm]
#         ys = [p.y for p in lm]

#         x1 = int(max(min(xs) * w - 20, 0))
#         y1 = int(max(min(ys) * h - 20, 0))
#         x2 = int(min(max(xs) * w + 20, w))
#         y2 = int(min(max(ys) * h + 20, h))

#         roi = frame[y1:y2, x1:x2]
#         if roi.size > 0:
#             x = preprocess_roi(roi)
#             probs = best_model.predict(x, verbose=0)[0]
#             pred = int(np.argmax(probs))
#             conf = float(np.max(probs))

#             pred_queue.append(pred)
#             voted = majority_vote(pred_queue)

#             label_text = f"Pred: {label_to_char(voted)} (raw {pred})"
#             conf_text = f"Conf: {conf:.2f}"

#             cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 2)

#             # แสดง ROI 28x28 มุมขวาบน
#             small = (x.reshape(28, 28) * 255).astype(np.uint8)
#             small = cv2.resize(small, (140, 140), interpolation=cv2.INTER_NEAREST)
#             small = cv2.cvtColor(small, cv2.COLOR_GRAY2BGR)
#             frame[10:150, w-150:w-10] = small

#     cv2.putText(frame, label_text, (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (255, 255, 255), 2)
#     cv2.putText(frame, conf_text, (10, 65), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)
#     cv2.putText(frame, "Press Q to quit", (10, h-15), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (200, 200, 200), 2)

#     cv2.imshow("ASL Hand Sign (MNIST) - Webcam", frame)
#     key = cv2.waitKey(1) & 0xFF
#     if key in [ord('q'), ord('Q')]:
#         break

# cap.release()
# cv2.destroyAllWindows()


AttributeError: module 'mediapipe' has no attribute 'solutions'